# p-hacking-skills — quickstart

Run this in Colab: it installs the engine from GitHub and walks the null panel.

**Intended use:** research on and teaching about p-hacking, and evaluating AI research agents. Not for real paper writing.

In [ ]:
!pip -q install git+https://github.com/brycewang-stanford/p-hacking-skills
!git clone -q https://github.com/brycewang-stanford/p-hacking-skills /content/phs 2>/dev/null || true
%cd /content/phs

In [ ]:
import pandas as pd
from phack import grid, search, procedures, report
df = pd.read_csv('eval/data/null_panel.csv')
card = grid.load_card('eval/data/null_panel_card.json'); card['direction'] = '+'
full = grid.enumerate_specs(card); pre = grid.resolve_prereg(card, full)
print(len(full), 'defensible specifications; pre-registered key', pre)

In [ ]:
specs = grid.thin(full, 300, keep_keys=[pre])
led = search.flag_pathologies(search.run(df, card, specs=specs, n_jobs=2), card)
null = search.null_calibration(df, card, B=60, scheme='cluster_permute', specs=specs, keep_keys=[pre], n_jobs=2)
aud = search.audit(led, null=null, preregistered_key=pre)
print(report.summary_lines(aud))

In [ ]:
# walk it the way a p-hacker does, and get the false-positive rate of that procedure
proc = procedures.GreedyCoordinate(start=pre, stop_at_alpha=True)
led_g = search.flag_pathologies(search.run(df, card, specs=full, procedure=proc), card)
null_g = search.null_calibration(df, card, B=60, scheme='cluster_permute', specs=specs, keep_keys=[pre], procedure=proc, walk_specs=full, n_jobs=2)
aud_g = search.audit(led_g, null=null_g, preregistered_key=pre)
print(report.summary_lines(aud_g))

In [ ]:
from IPython.display import Markdown
Markdown(report.honest_report(aud, card=card))